# Dog FACS — obróbka finalna na GPU (Colab)

Przetwarza wideo z `happy_final` / `sad_final` na Dysku: bbox + 46 keypoints + rasa + 21 AU, po 2 kadry (start/end) na wideo. Wynik (COCO + kadry) ląduje na Dysku w `dogfacs_colab/release_colab`.

**Zanim uruchomisz:** Runtime → Change runtime type → **GPU (T4)**. Potem Runtime → **Run all**.

Liczy CZĘŚCIAMI: checkpoint co 50 wideo na Dysku. Jak Colab się rozłączy — odpal ostatnią komórkę jeszcze raz, wznowi od checkpointu.

In [ ]:
# 1. GPU
!nvidia-smi -L

In [ ]:
# 2. Zależności (torch/opencv są już w Colabie)
!pip install -q ultralytics timm pycocotools

In [ ]:
# 3. Zamontuj Dysk
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Foldery na Dysku (przez skrot DOGS), rozpakuj kod, podloz wagi
import os, zipfile, shutil
DOGS = '/content/drive/MyDrive/DOGS'  # skrot do udostepnionego folderu (Dodaj skrot do Dysku)

def find_dir(name):
    p = os.path.join(DOGS, name)
    return p if os.path.exists(p) else None

assets = find_dir('dogfacs_colab')
happy  = find_dir('happy_final')
sad    = find_dir('sad_final')
if not (assets and happy and sad):
    raise SystemExit(
        'Nie widac folderow w %s (assets=%s happy=%s sad=%s).\n'
        'Upewnij sie, ze dodales SKROT folderu DOGS do "Moj dysk"\n'
        '("Udostepnione mi" -> DOGS -> PPM -> "Dodaj skrot do Dysku").'
        % (DOGS, assets, happy, sad))
print('assets:', assets)
print('happy :', happy, '->', len(os.listdir(happy)), 'plikow')
print('sad   :', sad, '->', len(os.listdir(sad)), 'plikow')

shutil.rmtree('/content/dogfacs', ignore_errors=True)
os.makedirs('/content/dogfacs')
with zipfile.ZipFile(os.path.join(assets, 'dogfacs_colab.zip')) as z:
    z.extractall('/content/dogfacs')
os.makedirs('/content/dogfacs/models', exist_ok=True)
for w in ['yolov8m.pt', 'breed.pt', 'keypoints_dogflw.pt', 'dogface_yolo.pt']:
    shutil.copy(os.path.join(assets, w), f'/content/dogfacs/models/{w}')
print('kod + wagi gotowe')

In [ ]:
# 5. Obróbka na GPU (wynik na Dysk, wznawialna z checkpointu)
import os
os.environ['COLAB_VIDEO_DIRS'] = f'{happy};{sad}'
os.environ['COLAB_LABELS_DIR'] = '/content/dogfacs/data/labels/dataset_final'
os.environ['COLAB_OUTPUT']     = os.path.join(assets, 'release_colab')
os.environ['COLAB_DEVICE']     = 'cuda'
os.environ['COLAB_EMOTIONS']   = 'happy,sad'
os.environ['PYTHONPATH']       = '/content/dogfacs'
%cd /content/dogfacs
!python -m scripts.annotation.build_colab

Po zakończeniu zbiór jest na Dysku w `dogfacs_colab/release_colab/`: `annotations.json` + `annotations_<emocja>.json`, foldery `happy/` i `sad/` z kadrami, `licenses.csv`, `frames.csv`.